In [0]:
%sql
SELECT * FROM spb.bronze_places
LIMIT 10;

In [0]:
import json, urllib.request
import pandas as pd
from pyspark.sql.functions import pandas_udf
from pyspark.sql import functions as F

latest = json.load(urllib.request.urlopen("https://stac.overturemaps.org/catalog.json"))["latest"]
base = f"s3://overturemaps-us-west-2/release/{latest}"
LANDING_DIR = "/Volumes/workspace/spb/landing"
MIN_CONFIDENCE = 0.4
H3_RES = 8

def flatten_places(df):
    return (
          df.withColumn("lon", F.col("bbox.xmin"))
          .withColumn("lat", F.col("bbox.ymin"))
          .drop("taxonomy", "bbox")
    )

def filter_places(df):
    return df.where(
        F.col("confidence") >= MIN_CONFIDENCE)

def dedupe(df):
    return df.dropDuplicates(["id"])


def add_h3(df):
    """Built-in H3 index. Note the argument order: longitude, then latitude."""
    return df.withColumn("h3_cell", F.expr(f"h3_longlatash3string(lon, lat, {H3_RES})"))